# H3 zonal stats from a GeoTIFF to Parquet

This notebook creates H3-based zonal statistics from a GeoTIFF and writes a Parquet file ready for ingestion (with a `hex_id` column and one or more metric columns).

**Inputs**
- GeoTIFF: `cy_frequency.tif`

**Outputs**
- CSV tiles (per H3 res-0 cell) in `data/cy_frequency_h3_csv/`
- Final Parquet in `data/cy_frequency_mean.parquet`


## 1) Configuration
Adjust H3 resolution, output column names, and other options here.

In [4]:
# Parameters
TIF_PATH = "cy_frequency.tif"
H3_LEVEL = 6
OUT_CSV_DIR = "data/cy_frequency_h3_csv"
OUT_PARQUET = "data/cy_frequency_mean.parquet"

# Choose output metric columns and names
# Options in the raw output: SUM, MIN, MAX, MEAN
OUTPUT_COLUMNS = {
    "MEAN": "cy_frequency_mean",
}


## 2) Imports and path setup
We re-use the project utilities in `notebooks/src`.

In [5]:
import os
import sys
import glob
import pandas as pd

# Ensure project helpers are importable
sys.path.insert(0, "../../src")

import h3_helper
import global_zonal
import GOSTrocks


## 3) Build H3 grid and run zonal stats
Use the shared land-masked H3 grid pickle (res 6) for consistency across layers, then run zonal stats over the GeoTIFF.


In [6]:
import os
import io
import re
from contextlib import redirect_stdout

import h3
from shapely.geometry import box
import geopandas as gpd
import rasterio
import GOSTrocks.rasterMisc as rMisc
import pandas as pd

os.makedirs(OUT_CSV_DIR, exist_ok=True)

# Load canonical H3 grid (polygons in EPSG:4326)
h3_0_list = h3_helper.generate_lvl0_lists(
    H3_LEVEL,
    return_gdf=True,
    buffer0=True,
    read_pickle=True,
    pickle_file="h0_dictionary_of_h6_geodata_frames_land.pickle",
)

csv_files = []
items = list(h3_0_list.items())
total = len(items)

with rasterio.open(TIF_PATH) as src:
    rast_crs = src.crs
    rast_bounds = box(*src.bounds)

    for i, (h0_code, gdf) in enumerate(items, start=1):
        out_csv = os.path.join(OUT_CSV_DIR, f"cy_frequency_{h0_code}.csv")

        # Reproject ONCE for this chunk (4326 -> raster CRS)
        gdf_rast = gdf.to_crs(rast_crs)

        # Bounds filter in raster CRS
        mask = gdf_rast.intersects(rast_bounds)
        n = int(mask.sum())

        print(f"[{i:3d}/{total}] h0={h0_code} intersecting_hexes={n} csvs_written={len(csv_files)}", flush=True)

        if n == 0:
            continue

        # Select from the already-reprojected GeoDataFrame (no 2nd to_crs)
        gdf_sel_rast = gdf_rast.loc[mask.values].copy()

        # Zonal stats already in raster CRS -> reProj=False
        # Capture occasional per-feature failures reported as: "Error processing <n>"
        _buf = io.StringIO()
        with redirect_stdout(_buf):
            res = rMisc.zonalStats(
                gdf_sel_rast,
                TIF_PATH,
                reProj=False,
            )
        failed_positions = {
            int(m.group(1))
            for m in re.finditer(r"Error processing\s+(\d+)", _buf.getvalue())
        }

        res = pd.DataFrame(res, columns=["SUM", "MIN", "MAX", "MEAN"])
        # shape_id should already be present from the pickle; if not, fallback:
        if "shape_id" in gdf_sel_rast.columns:
            ids = pd.Series(gdf_sel_rast["shape_id"].values)
        else:
            ids = pd.Series(gdf.loc[mask.values, "shape_id"].values)

        if failed_positions:
            bad_idx = [p - 1 for p in sorted(failed_positions) if p > 0]
            ids = ids.drop(index=bad_idx, errors="ignore").reset_index(drop=True)

        # Defensive fallback for any residual mismatch
        if len(ids) != len(res):
            n = min(len(ids), len(res))
            print(
                f"Warning: id/stat length mismatch for h0={h0_code} (ids={len(ids)}, stats={len(res)}). Truncating to {n}.",
                flush=True,
            )
            ids = ids.iloc[:n].reset_index(drop=True)
            res = res.iloc[:n].copy()

        res["id"] = ids.values

        res.to_csv(out_csv, index=False)
        csv_files.append(out_csv)

print(f"Done. Wrote {len(csv_files)} CSV files.", flush=True)

Loading pickle file h0_dictionary_of_h6_geodata_frames_land.pickle: it exists True
[  1/122] h0=8001fffffffffff intersecting_hexes=101610 csvs_written=0
[  2/122] h0=8003fffffffffff intersecting_hexes=102488 csvs_written=1
[  3/122] h0=8005fffffffffff intersecting_hexes=113227 csvs_written=2
[  4/122] h0=8007fffffffffff intersecting_hexes=117649 csvs_written=3
[  5/122] h0=8009fffffffffff intersecting_hexes=98041 csvs_written=4
[  6/122] h0=800bfffffffffff intersecting_hexes=117649 csvs_written=5
[  7/122] h0=800dfffffffffff intersecting_hexes=117649 csvs_written=6
[  8/122] h0=800ffffffffffff intersecting_hexes=117649 csvs_written=7
[  9/122] h0=8011fffffffffff intersecting_hexes=117649 csvs_written=8
[ 10/122] h0=8013fffffffffff intersecting_hexes=117649 csvs_written=9
[ 11/122] h0=8015fffffffffff intersecting_hexes=117649 csvs_written=10
[ 12/122] h0=8017fffffffffff intersecting_hexes=117649 csvs_written=11
[ 13/122] h0=8019fffffffffff intersecting_hexes=117649 csvs_written=12
[ 14/

In [7]:
import os
print(os.getcwd())

/Users/glevin/Documents/WorldBank/Space2Stats/notebooks/MP_SCRIPTS/TropicalCyclones


## 4) Combine CSVs and write Parquet
The output parquet includes a `hex_id` column and the selected metric columns.

In [8]:
csvs = glob.glob(os.path.join(OUT_CSV_DIR, "*.csv"))
if not csvs:
    raise FileNotFoundError(f"No CSVs found in {OUT_CSV_DIR}")

df = pd.concat((pd.read_csv(c) for c in csvs), ignore_index=True)

# Rename id -> hex_id for ingestion compatibility
if "id" in df.columns:
    df = df.rename(columns={"id": "hex_id"})

# Select and rename output columns
keep_cols = ["hex_id"] + list(OUTPUT_COLUMNS.keys())
missing = [c for c in keep_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns from zonal stats: {missing}")

df = df[keep_cols].rename(columns=OUTPUT_COLUMNS)


# Write parquet
os.makedirs(os.path.dirname(OUT_PARQUET), exist_ok=True)

# keep hex_id as string, force metrics to float
for col in OUTPUT_COLUMNS.values():  # e.g. ["cy_frequency_mean"]
    df[col] = pd.to_numeric(df[col], errors="coerce")
df.to_parquet(OUT_PARQUET, index=False)
OUT_PARQUET


'data/cy_frequency_mean.parquet'

## 5) Quick sanity checks

In [9]:
df.head()


,hex_id,cy_frequency_mean
0,869200007ffffff,0.0
1,86920000fffffff,0.0
2,869200017ffffff,0.0
3,86920001fffffff,0.0
4,869200027ffffff,0.0


In [13]:
import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import folium
import branca.colormap as cm

PARQUET = "data/cy_frequency_mean.parquet"
LEVEL = 6

# Read only required columns (faster)
df = pd.read_parquet(
    PARQUET,
    columns=["hex_id", "cy_frequency_mean"],
)

# -----------------------
# Bounding box (min_lon, min_lat, max_lon, max_lat)
# Bounding box for Japan (min_lon, min_lat, max_lon, max_lat)
bbox = (138.0, 34.5, 141.5, 37.5)  # Kanto
# # Kenya-ish bbox you can tweak:
# bbox = (33.5, -5.0, 42.5, 5.5)
min_lon, min_lat, max_lon, max_lat = bbox

# H3 polyfill of bbox (h3 wants (lat, lon))
bbox_poly = h3.LatLngPoly(
    [
        (min_lat, min_lon),
        (min_lat, max_lon),
        (max_lat, max_lon),
        (max_lat, min_lon),
    ]
)

bbox_cells = set(h3.polygon_to_cells(bbox_poly, LEVEL))
df_bbox = df[df["hex_id"].isin(bbox_cells)].copy()

print("rows in bbox:", len(df_bbox))
print("bbox non-NaN rows:", df_bbox["cy_frequency_mean"].notna().sum())

# Build polygons for ALL filtered hexes
def hex_to_poly(h):
    coords = h3.cell_to_boundary(h)  # (lat, lng)
    return Polygon([(lng, lat) for lat, lng in coords])

gdf_bbox = gpd.GeoDataFrame(
    df_bbox,
    geometry=[hex_to_poly(h) for h in df_bbox["hex_id"]],
    crs="EPSG:4326",
)

# A continuous colormap avoids the "bins" error from folium.Choropleth
# Your data seems to be in [0, 106]
colormap = cm.linear.YlOrRd_09.scale(0, 106)
colormap.caption = "Tropical cyclone frequency (mean)"

center_lat = (min_lat + max_lat) / 2.0
center_lon = (min_lon + max_lon) / 2.0
m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")

def style_fn(feat):
    v = feat["properties"].get("cy_frequency_mean", None)
    if v is None:
        return {"fillOpacity": 0.0, "weight": 0.0}
    # Clamp to [0,106] just in case
    v = max(0.0, min(106.0, float(v)))
    return {
        "fillColor": colormap(v),
        "color": "#000000",
        "weight": 0.2,
        "fillOpacity": 0.7,
    }

folium.GeoJson(
    gdf_bbox,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=["hex_id", "cy_frequency_mean"],
        aliases=["hex_id", "cyclone_frequency"],
        localize=True,
    ),
    name="hexes",
).add_to(m)

colormap.add_to(m)
folium.LayerControl().add_to(m)

m

rows in bbox: 3164
bbox non-NaN rows: 3164
